# E3-F1 — Audit reproductible de convergence

Analyse **sans réentraînement** des historiques contenus dans les archives E3-C, E3-D et E3-E3. Le critère « meilleur résultat dans les 10 dernières époques » est un **signal**, pas une preuve d’absence de convergence. L’arrêt anticipé explique pourquoi certains entraînements terminent avant 80 époques. Les comparaisons finales nécessitent un protocole de prolongation fixé sans consulter le test.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
OUTPUTS = ROOT / "outputs"
SOURCES = {
    "E3-C_local": OUTPUTS / "E3_10C_LSTM_TCN_multihorizon",
    "E3-D_GCN": OUTPUTS / "E3_10D_GCN_multihorizon",
    "E3-E3_multidevice": OUTPUTS / "E3_E3_multidevice_LSTM_TCN_105",
}
OUT = OUTPUTS / "E3_F1_AUDIT_CONVERGENCE"
OUT.mkdir(parents=True, exist_ok=True)

rows, curves = [], []
for exp, folder in SOURCES.items():
    if not folder.is_dir():
        matches = sorted(p for p in OUTPUTS.rglob("*") if p.is_dir() and p.name.lower() == folder.name.lower())
        if len(matches) == 1:
            folder = matches[0]
        else:
            raise FileNotFoundError(
                f"Dossier introuvable : {folder} ; dossiers candidats : "
                + str([p.name for p in OUTPUTS.iterdir() if p.is_dir() and "E3" in p.name])
            )
    files = sorted(folder.rglob("*_history.csv"))
    selected = []
    for path in files:
        # E3-C, E3-D, E3-E3 final runs; selection_history is excluded.
        m = re.search(r"h(\d{3})_(.+)_seed(42|123|2026)_history\.csv$", path.name)
        if m and "selection" not in path.name.lower():
            selected.append((path, int(m[1]), m[2], int(m[3])))
    print(f"{exp}: {len(selected)} historiques finaux dans {folder}")
    for path, h, model, seed in selected:
        df = pd.read_csv(path)
        if not {"epoch", "train_MSE", "validation_RMSE"}.issubset(df.columns):
            raise ValueError(f"Colonnes attendues absentes : {path}, {df.columns.tolist()}")
        if df.empty or not np.isfinite(df[["epoch", "train_MSE", "validation_RMSE"]].to_numpy()).all():
            raise ValueError(f"Historique vide ou non fini : {path}")
        v = df.validation_RMSE.to_numpy()
        epoch = df.epoch.to_numpy()
        best = int(np.argmin(v))
        w = min(10, len(v))
        lastw = float(np.mean(v[-w:]))
        prior = float(np.mean(v[-2*w:-w])) if len(v) >= 2*w else np.nan
        last_trend = lastw - prior
        rows.append(dict(
            experiment=exp, horizon_min=h, model=model, seed=seed,
            epochs_run=len(v), last_epoch=int(epoch[-1]),
            best_epoch=int(epoch[best]), best_validation_RMSE=float(v[best]),
            last_validation_RMSE=float(v[-1]),
            last_minus_best_RMSE=float(v[-1]-v[best]),
            last_10_mean_validation_RMSE=lastw,
            previous_10_mean_validation_RMSE=prior,
            final_window_delta_RMSE=last_trend,
            best_in_last_10=bool(best >= len(v)-w),
            hit_80_epoch_cap=bool(int(epoch[-1]) >= 80),
            relative_last10_change=float(last_trend/prior) if np.isfinite(prior) and prior else np.nan,
            initial_train_MSE=float(df.train_MSE.iloc[0]),
            last_train_MSE=float(df.train_MSE.iloc[-1])
        ))
        df["experiment"] = exp
        df["horizon_min"] = h
        df["model"] = model
        df["seed"] = seed
        curves.append(df)
summary=pd.DataFrame(rows).sort_values(['experiment','horizon_min','model','seed'])
assert len(summary)==(4*2*3+4*3*3+4*2*3),len(summary)
summary['diagnostic']=np.select([
 summary.hit_80_epoch_cap & summary.best_in_last_10 & (summary.final_window_delta_RMSE < -0.005),
 summary.hit_80_epoch_cap & summary.best_in_last_10,
 summary.hit_80_epoch_cap,
 summary.last_minus_best_RMSE>0.02],
 ['PRIORITY: improving at cap','CHECK: best near cap','CHECK: cap reached','CHECK: validation degradation'],default='No strong cap signal')
summary.to_csv(OUT/'01_diagnostics_per_run.csv',index=False)
pd.concat(curves,ignore_index=True).to_csv(OUT/'02_all_learning_curves.csv',index=False)
agg=summary.groupby(['experiment','model'],as_index=False).agg(runs=('seed','size'),hit_80=('hit_80_epoch_cap','sum'),best_last10=('best_in_last_10','sum'),mean_best_val_RMSE=('best_validation_RMSE','mean'),mean_last_minus_best=('last_minus_best_RMSE','mean'))
agg.to_csv(OUT/'00_summary.csv',index=False)
print(agg.to_string(index=False));print('\nDiagnostic counts\n',summary.diagnostic.value_counts().to_string());print('OUT',OUT)


In [ ]:
import matplotlib.pyplot as plt
for exp in summary.experiment.unique():
    subset=summary.loc[summary.experiment==exp]
    for model in subset.model.unique():
        fig,ax=plt.subplots(figsize=(9,4))
        curves_df=pd.concat(curves,ignore_index=True)
        d=curves_df[(curves_df.experiment==exp)&(curves_df.model==model)]
        for (h,seed),g in d.groupby(['horizon_min','seed']):
            ax.plot(g.epoch,g.validation_RMSE,alpha=.55,label=f'{h} min / seed {seed}')
        ax.set(title=f'{exp} — {model}',xlabel='Epoch',ylabel='Validation RMSE (°C)')
        ax.legend(ncol=4,fontsize=7)
        fig.tight_layout();fig.savefig(OUT/f'curve_{exp}_{model}.png',dpi=160)
        plt.show()


## Lecture scientifique

- **Plafond atteint** : 80 époques, sans conclusion automatique sur la convergence.
- **Meilleur score en fin d’entraînement** : possibilité d’amélioration si prolongation.
- **Tendance des fenêtres** : comparaison des moyennes de validation des dix dernières époques et des dix précédentes ; sensible au bruit.
- **Dégradation finale** : écart entre dernière et meilleure RMSE de validation, qui peut signaler un surapprentissage.

**Suite proposée :** protocole E3-F2 de prolongation à budget fixé, sélection sur validation seulement, avec checkpoints et comparaison à budget équivalent. Les résultats historiques doivent rester intacts.